# **Single-Cell RNA-Seq Analysis Project**

In this project, you will work with the `norman` dataset from the `perturbation_data_analysis` exercise:

In [ ]:
import os
os.environ["NUMBA_NUM_THREADS"] = "4"

import gc
import numpy as np
import scanpy as sc
import pertdata as pt
import matplotlib.pyplot as plt

from sklearn.covariance import MinCovDet
from scipy.stats import chi2
norman = pt.PertDataset(name="norman", cache_dir_path="data", silent=False)
print(norman)

Dataset already cached: /home/n4me/scrnaseq/src/data/norman
Loading: /home/n4me/scrnaseq/src/data/norman/norman/perturb_processed.h5ad
PertDataset object
    name: norman
    cache_dir_path: /home/n4me/scrnaseq/src/data
    path: /home/n4me/scrnaseq/src/data/norman
    adata: AnnData object with n_obs ✕ n_vars = 91205 ✕ 5045


In [ ]:
adata = norman.adata

# Identify mitochondrial genes (human dataset)
adata.var["mt"] = adata.var["gene_name"].str.startswith("MT-")

# Compute QC metrics
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt"],
    percent_top=[20],
    inplace=True
)

In [ ]:
adata.obs["log_total_counts"] = np.log1p(adata.obs["total_counts"])
adata.obs["log_n_genes"] = np.log1p(adata.obs["n_genes_by_counts"])


# QC

In [ ]:
qc_features = [
    "log_total_counts",
    "log_n_genes",
    "pct_counts_mt",
    "pct_counts_in_top_20_genes",
]

X_qc = adata.obs[qc_features].values

# Subsample for fitting (prevents crashes)
n_fit = min(5000, X_qc.shape[0])
idx = np.random.choice(X_qc.shape[0], n_fit, replace=False)

mcd = MinCovDet(random_state=0).fit(X_qc[idx])
dist2 = mcd.mahalanobis(X_qc)

threshold = chi2.ppf(0.99, df=X_qc.shape[1])
adata.obs["qc_outlier_mv"] = dist2 > threshold

# Clean memory
del X_qc, dist2
gc.collect()

In [ ]:
adata_filtered = adata[~adata.obs["qc_outlier_mv"]].copy()

del adata
gc.collect()

In [ ]:
sc.pl.violin(
    norman.adata,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    jitter=0.4,
    multi_panel=True
)
plt.close("all")

In [ ]:
sc.pl.violin(
    adata_filtered,
    ['n_genes_by_counts', 'total_counts', 'pct_counts_mt'],
    jitter=0.4,
    multi_panel=True
)
plt.close("all")

In [ ]:
sc.pl.scatter(
    adata_filtered,
    "total_counts",
    "n_genes_by_counts",
    color="pct_counts_mt"
)
plt.close("all")

In [ ]:
print("Final cell count:", adata_filtered.n_obs)
print("Final gene count:", adata_filtered.n_vars)

## Normalizing

In [ ]:
adata_filtered.raw = adata_filtered.copy()

# Library size normalization
sc.pp.normalize_total(adata_filtered, target_sum=1e4)

# Log-transformation
sc.pp.log1p(adata_filtered)

In [ ]:
sc.pp.highly_variable_genes(
    adata_filtered,
    flavor="seurat_v3",
    n_top_genes=2000,
)

sc.pl.highly_variable_genes(adata_filtered)
adata_filtered.shape

In [ ]:
adata_hiVariGenes = adata_filtered[:, adata_filtered.var.highly_variable]
print(f"Shape of highly variable df: {adata_hiVariGenes.shape}")
print(f"Shape of highly only filtered df: {adata_filtered.shape}")


In [ ]:
# sc.pp.scale(adata_hiVariGenes, max_value=10)

In [ ]:
sc.tl.pca(adata_hiVariGenes, n_comps=2)
sc.pl.pca(
    adata_hiVariGenes,
    color=['total_counts', 'pct_counts_mt'],
    components=['1,2'],
    size=20
)

sc.tl.pca(adata_hiVariGenes, n_comps=3)
sc.pl.pca(
    adata_hiVariGenes,
    color=['total_counts', 'pct_counts_mt'],
    components=['1,2', '1,3', '2,3'],
    size=20
)

In [ ]:
print(adata_hiVariGenes.obs.columns)


sc.pp.regress_out(adata_hiVariGenes, ['total_counts', 'pct_counts_mt'])

In [ ]:
adata_hiVariGenes.obs["condition_fixed"] = adata_hiVariGenes.obs["condition"]


sc.tl.pca(adata_hiVariGenes, n_comps=2)
sc.pl.pca(
    adata_hiVariGenes,
    components=['1,2'],
    size=20,
    title='PCA - Two primary components after regressing out unwanted factors (colored by total count)', 
    color='total_counts', 
    palette='Set1',
    legend_fontsize='xx-small'
)

sc.tl.pca(adata_hiVariGenes, n_comps=2)
sc.pl.pca(
    adata_hiVariGenes,
    components=['1,2'],
    size=20,
    title='PCA - Two primary components after regressing out unwanted factors', 
    color='condition_fixed', 
    palette='Set1',
    legend_fontsize='xx-small'
)

sc.tl.pca(adata_hiVariGenes, n_comps=3)
sc.pl.pca(
    adata_hiVariGenes,
    components=['1,2,3'],
    size=20, color='condition_fixed',
    title='PCA - Three primary components after regressing out unwanted factors',
    projection='3d', palette='Set1',
    legend_fontsize='xx-small'
)

In [ ]:
sc.tl.pca(adata_hiVariGenes, n_comps=50)
sc.pl.pca(
    adata_hiVariGenes,
    components=['1,2'],
    size=20,
    title='PCA - First two primary components of 50 PC´s after regressing out unwanted factors',
)

In [ ]:
adata_cp = adata_hiVariGenes  
ctrl_labels = ["ctrl", "control", "NTC"]  
adata_cp.obs["cond_group"] = np.where(
    adata_cp.obs["condition_fixed"].isin(ctrl_labels),
    "control",
    "perturbed"
)

sc.pl.pca(
    adata_hiVariGenes,
    components=['1,2'],
    color="cond_group",
    title='PCA - First two primary components with coloring of control vs pertubed',
    size=20
)

In [ ]:
sc.pl.pca_variance_ratio(adata_hiVariGenes, log=True, n_pcs=50)

In [ ]:
sc.pp.neighbors(adata_hiVariGenes, n_pcs=50, n_neighbors=15) 

sc.tl.leiden(adata_hiVariGenes, resolution=0.6)

In [ ]:
sc.tl.umap(adata_hiVariGenes)

sc.pl.umap(adata_hiVariGenes, color='leiden', legend_loc='on data', size=5, title='UMAP of Leiden clusters')
sc.pl.umap(adata_hiVariGenes, color='condition_fixed', size=5,  palette='Set1', frameon=False, title='UMAP of perturbation/control', legend_fontsize='xx-small')

# Statistik
print(adata_hiVariGenes.obs['leiden'].value_counts())
print(f"Anzahl Cluster: {len(adata_hiVariGenes.obs['leiden'].unique())}")

### Plotting UMAP of controled vs pertubed Genes 

In [ ]:
adata_cp = adata_hiVariGenes  
ctrl_labels = ["ctrl", "control", "NTC"]  
adata_cp.obs["cond_group"] = np.where(
    adata_cp.obs["condition_fixed"].isin(ctrl_labels),
    "control",
    "perturbed"
)

sc.tl.umap(adata_hiVariGenes)

sc.pl.umap(adata_hiVariGenes, color="cond_group", 
           size=5, palette=["#1f77b4", "#ff7f0e"], frameon=False, title='Controled vs perturbed genes on basis of UMAP')


### t-SNE visualization

In [ ]:
sc.pp.neighbors(adata_hiVariGenes, n_neighbors=15, n_pcs=50)

sc.tl.tsne(
    adata_hiVariGenes,
    n_pcs=50,                    
    learning_rate=1000,          
    early_exaggeration=12,       
    n_jobs=-1,                   
    random_state=0              
)

In [ ]:
# Plot
threshold = 600
pert_counts = adata_hiVariGenes.obs['condition_fixed'].value_counts()
top_perts = pert_counts[pert_counts > threshold].index.tolist()

adata_hiVariGenes.obs['pert_threshold'] = adata_hiVariGenes.obs['condition_fixed'].apply(
    lambda x: x if x in top_perts else 'Other'
)


adata_hiVariGenes.obs['pert_threshold'] = adata_hiVariGenes.obs['pert_threshold'].astype('category')
categories = [cat for cat in adata_hiVariGenes.obs['pert_threshold'].cat.categories if cat not in ['ctrl', 'Other']]

if 'ctrl' in adata_hiVariGenes.obs['pert_threshold'].cat.categories:
    categories = categories + ['ctrl']
categories = categories + ['Other']
adata_hiVariGenes.obs['pert_threshold'] = adata_hiVariGenes.obs['pert_threshold'].cat.reorder_categories(categories)

import matplotlib.pyplot as plt
n_perts_without_ctrl = len([p for p in top_perts if p != 'ctrl'])
colors_hsv = plt.cm.hsv(np.linspace(0, 1, n_perts_without_ctrl))
color_list = [plt.matplotlib.colors.rgb2hex(c) for c in colors_hsv]

if 'ctrl' in top_perts:
    color_list.append('#000000')  
color_list.append('#808080')  

print(f"Schwellenwert: >{threshold} Zellen")
print(f"Anzahl Perturbationen über Schwellenwert: {len(top_perts)}")
print(f"Perturbationen (ohne ctrl): {[p for p in top_perts if p != 'ctrl']}")
print(f"Anzahl 'Other': {(adata_hiVariGenes.obs['pert_threshold'] == 'Other').sum()}")

sc.settings.set_figure_params(dpi=100, figsize=(10, 8))
sc.settings.file_format_figs = 'png'

fig, ax = plt.subplots(figsize=(10, 8), dpi=100)

tsne_coords = adata_hiVariGenes.obsm['X_tsne']

colors_mapped = []
categories_list = list(adata_hiVariGenes.obs['pert_threshold'].cat.categories)
for pert in adata_hiVariGenes.obs['pert_threshold']:
    idx = categories_list.index(pert)
    colors_mapped.append(color_list[idx])

scatter = ax.scatter(
    tsne_coords[:, 0], 
    tsne_coords[:, 1],
    c=colors_mapped,
    s=1,
    alpha=0.8
)

from matplotlib.lines import Line2D
legend_elements = [Line2D([0], [0], marker='o', color='w', 
                          markerfacecolor=color_list[i], markersize=8, label=cat) 
                   for i, cat in enumerate(categories_list)]
ax.legend(handles=legend_elements, loc='center left', bbox_to_anchor=(1, 0.5), 
          frameon=False, fontsize=9)

ax.set_xlabel('t-SNE 1', fontsize=12)
ax.set_ylabel('t-SNE 2', fontsize=12)
ax.set_title('')

ax.set_facecolor('#f0f5f9')

ax.grid(True, color='white', linewidth=1, alpha=0.7, zorder=0)

scatter = ax.scatter(
    tsne_coords[:, 0], 
    tsne_coords[:, 1],
    c=colors_mapped,
    s=1,
    alpha=0.8,
    zorder=2
)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.5)
    spine.set_edgecolor('white')

ax.tick_params(axis='both', which='major', labelsize=10, length=0, width=0, 
               labelcolor='black')

fig.savefig('figures/tsne_threshold.png', dpi=300, bbox_inches='tight')
print("Plot gespeichert als: figures/tsne_threshold.png")


In [ ]:
# 1) Compute markers per Leiden cluster
sc.tl.rank_genes_groups(
    adata_hiVariGenes,
    groupby="leiden",
    method="wilcoxon",
    n_genes=50
)

# 2) Quick overview table
sc.pl.rank_genes_groups(adata_hiVariGenes, n_genes=20, sharey=False)

# 3) Dotplot / heatmap for top markers
sc.pl.rank_genes_groups_dotplot(
    adata_hiVariGenes,
    n_genes=10,
    standard_scale="var"
)
# or:
sc.pl.rank_genes_groups_heatmap(
    adata_hiVariGenes,
    n_genes=10,
    standard_scale="var",
    swap_axes=True
)

In [ ]:
# Show markers for specific clusters
sc.pl.rank_genes_groups(adata_hiVariGenes, groups=["0","1","2"], n_genes=15)


In [ ]:
sc.pl.rank_genes_groups_heatmap(adata_hiVariGenes,
                                n_genes=3,
                                groupby='leiden',
                                figsize=(14, 6),
                                show_gene_labels=True,
                                cmap= 'coolwarm',
                                standard_scale='var')

In [ ]:
adata_hiVariGenes.X = np.maximum(adata_hiVariGenes.X, 0)
sc.tl.rank_genes_groups(
    adata_hiVariGenes,
    groupby='leiden',
    method='wilcoxon'
)

In [ ]:
clusterplot = sc.pl.rank_genes_groups_dotplot(adata_hiVariGenes, gene_symbols="gene_name", standard_scale="var", n_genes=3, groupby='leiden',  title='Top 3 expressed genes per leiden cluster')

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata_hiVariGenes, n_genes=3, groupby='condition_fixed',title='Top 3 expressed genes per leiden cluster in reference to respective perturbations')

In [ ]:
sc.tl.rank_genes_groups(
    adata_hiVariGenes,
    groupby='condition_fixed',
    method='wilcoxon'
)

sc.pl.rank_genes_groups_dotplot(adata_hiVariGenes, n_genes=3, groupby='condition_fixed')

In [ ]:


sc.pl.violin(adata_hiVariGenes, keys=["n_genes", "total_counts", "pct_counts_mt"], groupby="leiden", stripplot=False, inner="box")